In [1]:
"""
Simple helper to download Landsat 8/9 Collection 2 Level-2 products
(30m SR bands 1-7, ST_B10, QA + aux files) from the usgs-landsat S3 bucket.

Prerequisites:
- `pip install boto3`
- Configure AWS credentials (e.g., `aws configure`)
- S3 bucket is requester-pays, so we pass RequestPayer='requester'.

Usage:
    from landsat_c2_downloader import download_landsat_scene

    scene_id = "LC08_L2SP_172057_20210101_20210308_02_T1"
    download_landsat_scene(scene_id, "/data/yuyao/landsat_raw")
"""

import os
from dataclasses import dataclass
from typing import List, Dict

import boto3
from botocore.exceptions import ClientError


BUCKET_NAME = "usgs-landsat"
COLLECTION_PREFIX = "collection02/level-2/standard/oli-tirs"
AWS_REGION = "us-west-2"  # Landsat bucket region


@dataclass
class LandsatSceneLocation:
    scene_id: str
    year: str
    path: str
    row: str

    @property
    def s3_prefix(self) -> str:
        """
        S3 prefix for this scene under the usgs-landsat bucket.

        s3://usgs-landsat/collection02/level-2/standard/oli-tirs/<year>/<path>/<row>/<scene_id>/
        """
        return f"{COLLECTION_PREFIX}/{self.year}/{self.path}/{self.row}/{self.scene_id}"


def parse_scene_id(scene_id: str) -> LandsatSceneLocation:
    """
    Parse a Landsat Collection 2 Level-2 scene ID.

    Example scene_id:
        LC08_L2SP_172057_20210101_20210308_02_T1

    Format:
        LXSS_LLLL_PPPRRR_YYYYMMDD_yyyymmdd_CC_TX

    We need:
        year   = YYYY
        path   = PPP
        row    = RRR
    """
    parts = scene_id.split("_")
    if len(parts) < 4:
        raise ValueError(f"Invalid scene_id format: {scene_id}")

    pprrr = parts[2]
    if len(pprrr) != 6 or not pprrr.isdigit():
        raise ValueError(f"Invalid path/row in scene_id: {scene_id}")

    path = pprrr[:3]
    row = pprrr[3:]
    year = parts[3][:4]

    return LandsatSceneLocation(scene_id=scene_id, year=year, path=path, row=row)


def build_required_filenames(scene_id: str) -> Dict[str, List[str]]:
    """
    Given a scene_id, return the list of filenames we want to download
    from the scene directory.

    - 30 m Surface Reflectance bands: SR_B1–SR_B7
    - 30 m Surface Temperature: ST_B10
    - QA / auxiliary:
        QA_PIXEL, QA_RADSAT, SR_QA_AEROSOL, ST_QA
        ANG.txt, MTL.txt
    """
    # SR bands 1-7
    sr_bands = [f"{scene_id}_SR_B{b}.TIF" for b in range(1, 8)]

    # Surface temperature band
    st_band = [f"{scene_id}_ST_B10.TIF"]

    # QA rasters
    qa_files = [
        f"{scene_id}_QA_PIXEL.TIF",
        f"{scene_id}_QA_RADSAT.TIF",
        f"{scene_id}_SR_QA_AEROSOL.TIF",
        f"{scene_id}_ST_QA.TIF",
    ]

    # Auxiliary text files
    aux_files = [
        f"{scene_id}_ANG.txt",
        f"{scene_id}_MTL.txt",
    ]

    return {
        "sr": sr_bands,
        "st": st_band,
        "qa": qa_files,
        "aux": aux_files,
    }


def get_s3_client(region_name: str = AWS_REGION):
    """
    Create a boto3 S3 client. Assumes AWS credentials are configured.
    """
    return boto3.client("s3", region_name=region_name)


def download_single_object(
    s3_client,
    key: str,
    local_path: str,
    request_payer: str = "requester",
) -> bool:
    """
    Download a single S3 object (key) to local_path.

    Returns True if downloaded or already exists, False if object not found.
    """
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    if os.path.exists(local_path):
        # Already downloaded
        print(f"[skip] {local_path} already exists.")
        return True

    try:
        print(f"[download] s3://{BUCKET_NAME}/{key} -> {local_path}")
        s3_client.download_file(
            BUCKET_NAME,
            key,
            local_path,
            ExtraArgs={"RequestPayer": request_payer},
        )
        return True
    except ClientError as e:
        # NotFound or permission issues
        error_code = e.response.get("Error", {}).get("Code", "")
        if error_code == "404" or "Not Found" in str(e):
            print(f"[missing] s3://{BUCKET_NAME}/{key} (404)")
            return False
        print(f"[error] downloading {key}: {e}")
        return False


def download_landsat_scene(
    scene_id: str,
    output_root: str,
    request_payer: str = "requester",
) -> Dict[str, List[str]]:
    """
    Download required files for a single Landsat 8/9 C2 L2 scene
    from the usgs-landsat requester-pays bucket.

    Parameters
    ----------
    scene_id : str
        e.g. "LC08_L2SP_172057_20210101_20210308_02_T1"
    output_root : str
        Local directory under which a subfolder with the scene_id
        will be created.
    request_payer : str
        "requester" is required for requester-pays buckets.

    Returns
    -------
    downloaded : Dict[str, List[str]]
        Dictionary with keys "sr", "st", "qa", "aux" and values
        being lists of local file paths that were successfully downloaded.
    """
    loc = parse_scene_id(scene_id)
    s3_prefix = loc.s3_prefix  # collection02/level-2/standard/oli-tirs/...

    required = build_required_filenames(scene_id)
    s3_client = get_s3_client()

    scene_local_dir = os.path.join(output_root, scene_id)
    os.makedirs(scene_local_dir, exist_ok=True)

    downloaded: Dict[str, List[str]] = {"sr": [], "st": [], "qa": [], "aux": []}

    for group_name, filenames in required.items():
        for fname in filenames:
            s3_key = f"{s3_prefix}/{fname}"
            local_path = os.path.join(scene_local_dir, fname)

            ok = download_single_object(
                s3_client=s3_client,
                key=s3_key,
                local_path=local_path,
                request_payer=request_payer,
            )
            if ok:
                downloaded[group_name].append(local_path)

    return downloaded


if __name__ == "__main__":
    # Simple manual test / demo
    test_scene_id = "LC08_L2SP_172057_20210101_20210308_02_T1"
    out_dir = "./landsat_download_test"

    result = download_landsat_scene(test_scene_id, out_dir)
    print("Downloaded files:")
    for k, v in result.items():
        print(f"  {k}:")
        for p in v:
            print(f"    - {p}")

[download] s3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/172/057/LC08_L2SP_172057_20210101_20210308_02_T1/LC08_L2SP_172057_20210101_20210308_02_T1_SR_B1.TIF -> ./landsat_download_test/LC08_L2SP_172057_20210101_20210308_02_T1/LC08_L2SP_172057_20210101_20210308_02_T1_SR_B1.TIF
[download] s3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/172/057/LC08_L2SP_172057_20210101_20210308_02_T1/LC08_L2SP_172057_20210101_20210308_02_T1_SR_B2.TIF -> ./landsat_download_test/LC08_L2SP_172057_20210101_20210308_02_T1/LC08_L2SP_172057_20210101_20210308_02_T1_SR_B2.TIF
[download] s3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/172/057/LC08_L2SP_172057_20210101_20210308_02_T1/LC08_L2SP_172057_20210101_20210308_02_T1_SR_B3.TIF -> ./landsat_download_test/LC08_L2SP_172057_20210101_20210308_02_T1/LC08_L2SP_172057_20210101_20210308_02_T1_SR_B3.TIF
[download] s3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/172/057/LC08_L2SP_172057_20210101_20210308_02_T1/LC

In [3]:
from landsat_stac_utils import fetch_landsat_items
# Translated comment
from datetime import datetime, timezone, timedelta

plume_bounds = [-103.6, 31.9, -103.4, 32.1]
event_dt = datetime(2022, 7, 15, 18, 47, tzinfo=timezone.utc)

items = fetch_landsat_items(plume_bounds, event_dt - timedelta(days=7), event_dt + timedelta(days=7))
print(len(items))
for it in items[:5]:
    print(it["scene_id"], it["acq_time"])


2
LC09_L2SP_031038_20220711_20230407_02_T1_SR 2022-07-11 17:27:08.719945+00:00
LC08_L2SP_031038_20220719_20220725_02_T1_SR 2022-07-19 17:27:36.333305+00:00
